<a href="https://colab.research.google.com/github/manga-sonta/SAM_Household_Masks/blob/main/colab_run_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM + augmentation consistency pipeline on Colab

Runs the full pipeline (no SAM3D) on 100–150 images:
1. Clone repo (or use your GitHub URL)
2. Install deps + download SAM checkpoint
3. Download MIT Indoor, sample 100–150 images (3 classes)
4. Run SAM on originals → masks + metadata
5. Generate augmented images (even: hflip / rotation / resize / blur)
6. Run SAM on augmented images
7. Evaluate consistency → analysis → plots

**Runtime:** GPU recommended (faster SAM). Results can be zipped and downloaded.

## 1. Clone repo and go to project dir

Push your `sam-household-masks` code to a GitHub repo, then paste the clone URL below (use HTTPS, e.g. `https://github.com/USERNAME/sam-household-masks.git`). If you already have the repo elsewhere, you can skip clone and set `PROJECT_DIR` manually.

In [1]:
REPO_URL = "https://github.com/manga-sonta/SAM_Household_Masks.git"

import os

!git clone --depth 1 $REPO_URL

PROJECT_DIR = "/content/SAM_Household_Masks"

os.chdir(PROJECT_DIR)
print("Project dir:", os.getcwd())

fatal: destination path 'SAM_Household_Masks' already exists and is not an empty directory.
Project dir: /content/SAM_Household_Masks


In [2]:
!ls

analyze_consistency.py	  evaluate_consistency.py  requirements.txt
batch_sam_masks.py	  generate_augmented.py    run_sam3d.py
colab_run_pipeline.ipynb  images		   sam_output
doc			  plot_consistency.py
download_mit_indoor.py	  README.md


## 2. Install dependencies and SAM checkpoint

In [3]:
!pip install -q torch torchvision numpy opencv-python Pillow tqdm scipy matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

  Preparing metadata (setup.py) ... done


In [ ]:
# Download SAM checkpoint (ViT-B, ~375 MB)
CHECKPOINT = "sam_vit_b_01ec64.pth"
if not os.path.isfile(os.path.join(PROJECT_DIR, CHECKPOINT)):
    !wget -q -O {CHECKPOINT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Downloaded", CHECKPOINT)
else:
    print("Checkpoint already present:", CHECKPOINT)

## 3. Download MIT Indoor and sample 100–150 images (3 classes)

In [ ]:
import os
import glob

IMAGE_DIR = os.path.join(PROJECT_DIR, "images", "mit_indoor_subset")

images = glob.glob(os.path.join(IMAGE_DIR, "*.*"))
print("Images found:", len(images))

print("Example images:")
print(images[:5])

## 4. Run SAM on original images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_subset --output sam_output/original --checkpoint /content/sam_vit_b_01ec64.pth --model vit_b

## 5. Generate augmented images (even: hflip, rotation, resize, blur)

In [ ]:
!python generate_augmented.py --images images/mit_indoor_subset --output-dir images/mit_indoor_aug --manifest sam_output/aug_manifest.json --augs-per-image 1

## 6. Run SAM on augmented images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_aug --output sam_output/augmented --checkpoint sam_vit_b_01ec64.pth --model vit_b

## 7. Evaluate consistency, analyze, plot

In [ ]:
!python evaluate_consistency.py --original-output sam_output/original --augmented-output sam_output/augmented --manifest sam_output/aug_manifest.json --results sam_output/consistency_results.json

In [ ]:
!python analyze_consistency.py --results sam_output/consistency_results.json --original-metadata sam_output/original/metadata --output sam_output/analysis.json

In [ ]:
!python plot_consistency.py --analysis sam_output/analysis.json --output-dir sam_output/plots

## 8. Zip results and download (optional)

In [ ]:
os.chdir(PROJECT_DIR)
!zip -r sam_pipeline_results.zip sam_output/ images/mit_indoor_subset/ images/mit_indoor_aug/ -x "*.pyc" 2>/dev/null
from google.colab import files
zip_path = os.path.join(PROJECT_DIR, "sam_pipeline_results.zip")
if os.path.isfile(zip_path):
    files.download(zip_path)
    print("Download started: sam_pipeline_results.zip")
else:
    print("Zip not found; download sam_output/ and images/ from the file browser.")